# Guia Completo: Tradução, Remoção de Legendas e Lip-Sync com Clonagem de Voz (XTTS)

Este notebook contém todos os passos para pegar um vídeo com legendas hardcoded, remover/borrar as legendas, traduzir o áudio por segmentos para o inglês mantendo a sincronia (usando XTTS-v2), e aplicar lip-sync com Wav2Lip.

**Importante:** Ative a GPU A100 (ou V100/T4) em `Ambiente de Execução` -> `Alterar tipo de ambiente de execução`.

## Passo 1: Instalação das Dependências

Instalando bibliotecas de processamento de áudio, Whisper, TTS (Coqui) e Wav2Lip.

In [ ]:
!apt-get install -y ffmpeg
!pip install -U openai-whisper pydub
!pip install TTS

!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip
!pip install -r requirements.txt
!pip install librosa==0.8.0

# Baixando os pesos do Wav2Lip automaticamente
!wget 'https://iiitaphyd-my.sharepoint.com/personal/radrabha_m_research_iiit_ac_in/_layouts/15/download.aspx?share=EdjI7bZlgApMqsKSYmVpmOoBdPNvnVGFlYn2SmKk4xqRvg' -O checkpoints/wav2lip_gan.pth
!wget 'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth' -O face_detection/detection/sfd/s3fd.pth
%cd ..

import os
print("Dependências instaladas com sucesso!")

## Passo 2: Upload do Vídeo

In [ ]:
import os
from google.colab import files

os.makedirs("projeto_traducao", exist_ok=True)
%cd projeto_traducao

print("Selecione seu arquivo de vídeo (ex: EP 05 - LARA E PEDRO.mov):")
uploaded = files.upload()
video_original = list(uploaded.keys())[0]
video_sem_legenda = "video_sem_legenda.mp4"
print(f"Vídeo '{video_original}' enviado com sucesso!")

## Passo 3: Remoção/Desfoque da Legenda

Desfocando a parte inferior do vídeo para remover as legendas originais.

In [ ]:
# boxblur=20:5. Ajuste y=ih-150 conforme a altura da legenda
comando_ffmpeg = f'ffmpeg -i "{video_original}" -filter_complex "[0:v]crop=iw:150:0:ih-150,boxblur=20:5[bg];[0:v][bg]overlay=0:H-h" -c:a copy "{video_sem_legenda}"'
os.system(comando_ffmpeg)
print(f"Vídeo salvo como {video_sem_legenda}")

## Passo 4: Transcrição, Tradução e Sincronização (Whisper + XTTS-v2)

Para manter o Lip-Sync correto, precisamos extrair os tempos de fala de cada frase, traduzir com o Whisper e gerar a voz com XTTS **por segmento**, encaixando na linha do tempo original.

In [ ]:
import whisper
import subprocess
from pydub import AudioSegment
import torch
from TTS.api import TTS

audio_original = "audio_original.wav"
audio_ingles = "audio_ingles.wav"
subprocess.run(['ffmpeg', '-i', video_original, '-q:a', '0', '-map', 'a', audio_original, '-y'])

print("Carregando Whisper...")
modelo_whisper = whisper.load_model("large")

print("Transcrevendo e mapeando tempos...")
resultado = modelo_whisper.transcribe(audio_original, task="translate")

print("Carregando TTS (XTTS-v2)...")
device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

audio_completo_referencia = AudioSegment.from_wav(audio_original)
audio_final_pydub = AudioSegment.silent(duration=len(audio_completo_referencia))

print("Gerando segmentos de voz sincronizados...")
for segment in resultado["segments"]:
    start_ms = int(segment["start"] * 1000)
    end_ms = int(segment["end"] * 1000)
    texto_traduzido = segment["text"]
    
    # Extrair um pedaço de áudio como referencia para clonar
    ref_start = start_ms
    ref_end = min(len(audio_completo_referencia), start_ms + 5000)
    audio_ref = audio_completo_referencia[ref_start:ref_end]
    audio_ref.export("temp_ref.wav", format="wav")
    
    # Gerar a fala clonada para o segmento
    try:
        tts.tts_to_file(
            text=texto_traduzido,
            speaker_wav="temp_ref.wav",
            language="en",
            file_path="temp_out.wav"
        )
        
        if os.path.exists("temp_out.wav"):
            segmento_gerado = AudioSegment.from_wav("temp_out.wav")
            # Encaixar o segmento gerado no tempo original
            audio_final_pydub = audio_final_pydub.overlay(segmento_gerado, position=start_ms)
    except Exception as e:
        print(f"Erro ao gerar segmento: {e}")

audio_final_pydub.export(audio_ingles, format="wav")
print("Áudio sincronizado em inglês gerado com sucesso!")

## Passo 5: Lip-Sync (Wav2Lip)

In [ ]:
%cd /content/Wav2Lip

video_input = f"/content/projeto_traducao/{video_sem_legenda}"
audio_input = f"/content/projeto_traducao/audio_ingles.wav"
video_final = "/content/projeto_traducao/video_final_lipsync.mp4"

comando_wav2lip = f"python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face '{video_input}' --audio '{audio_input}' --outfile '{video_final}'"
os.system(comando_wav2lip)

%cd /content/projeto_traducao
print("Lip-Sync Concluído!")

## Passo 6: Download do Vídeo Final

In [ ]:
from google.colab import files
files.download(video_final)